In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import sys

from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts import (
    generate_dataset,
    optuna_search,
    predictions,
    training,
)
from experiments.plotting import _plot_SRE_distribution, view_correlation

from src.utils import configure_logger
from GNN.training.utils import to_scalar
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
from GNN.training.utils import collect_dataset_indices
from GNN.physics_aware_NN import ShardedQuantumCircuitGraphDataset

In [5]:
def dataset_to_dataframe(dataset):
    data = []
    keep_fields = ["n_qubits", "n_layers", "sre", "seed", "family"]
    for item in dataset:
        row = {}
        for field in keep_fields:
            if not hasattr(item, field):
                continue

            value = getattr(item, field)
            if field == "family":
                # Keep categorical label as text (e.g., "clifford")
                row[field] = str(value)
            else:
                row[field] = to_scalar(value)
        data.append(row)

    df = pd.DataFrame(data)

    int_cols = ["n_qubits", "n_layers", "seed"]
    for col in int_cols:
        if col in df.columns:
            df[col] = df[col].astype(int)

    float_cols = ["sre"]
    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype(float)

    if "family" in df.columns:
        df["family"] = df["family"].astype(str)

    return df

In [6]:
target_variant = "sre"
family = "random"
# regime_type = "identity_like"
regime_type = "saturated"

index_path = collect_dataset_indices(
    "../outputs/data/datasets_SRE",
    family=family,
)
print(f"Collected {len(index_path)} dataset indices for family '{family}' and regime '{regime_type}'")
base_dataset = ShardedQuantumCircuitGraphDataset(
    index_paths=index_path,
    target_variant=target_variant,
    split="all",
    cache_size=64,
)
df = dataset_to_dataframe(base_dataset)

plot_df = df[
    (df["sre"].notna())
]

avg_df = (
    plot_df
    .groupby(["n_qubits", "n_layers", "family"])
    .agg(
        sre_mean=("sre", "mean"),
        sre_std=("sre", "std"),
    )
    .reset_index()
)

avg_df.to_csv(f"final/data/avg_sre_{family}.csv", index=False)

Collected 1 dataset indices for family 'random' and regime 'saturated'


In [10]:
df.head()

,n_qubits,n_layers,sre,seed,family
0,4,1,0.202442,22550440,random
1,4,1,0.000129,25064164,random
2,4,1,0.682258,35019855,random
3,4,1,0.412548,35413194,random
4,4,1,0.374010,42486639,random


In [9]:
df_random = df[df["family"] == "random"].copy()
print(df_random["sre"].describe())
print("target_variant:", df_random["sre"].var())
print("Std:", df_random["sre"].std())

df_random.groupby("n_qubits")["sre"].agg(
    ["count", "mean", "std", "var"],
)

count    3.570000e+04
mean     3.400162e+00
std      2.365550e+00
min     -2.562741e-15
25%      1.638032e+00
50%      3.153243e+00
75%      5.348991e+00
max      8.013436e+00
Name: sre, dtype: float64
target_variant: 5.595829062325704
Std: 2.365550477653289


,count,mean,std,var
n_qubits,,,,
4,8925,1.631783,0.821712,0.675211
6,8925,2.799404,1.459088,2.128938
8,8925,3.984315,2.131372,4.542746
10,8925,5.185146,2.832321,8.022040
12,0,NaN,NaN,NaN
14,0,NaN,NaN,NaN
16,0,NaN,NaN,NaN
18,0,NaN,NaN,NaN
20,0,NaN,NaN,NaN


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_true = predictions_df["true"].to_numpy()
y_pred = predictions_df["pred"].to_numpy()

mean_pred = np.full_like(y_true, y_true.mean())

print("Model MSE:", mean_squared_error(y_true, y_pred))
print("Mean baseline MSE:", mean_squared_error(y_true, mean_pred))

print("Model MAE:", mean_absolute_error(y_true, y_pred))
print("Mean baseline MAE:", mean_absolute_error(y_true, mean_pred))

print("Model R2:", r2_score(y_true, y_pred))